## Silver Layer

## Cleaning and Normalizing Customers table

In [0]:
%sql
USE CATALOG course_training_catalog;
CREATE SCHEMA IF NOT EXISTS silver_olist;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW customer_base AS
SELECT DISTINCT
  customer_id,
  customer_unique_id,
  customer_zip_code_prefix,
  customer_city,
  customer_state
FROM course_training_catalog.bronze_olist.customers
WHERE customer_id IS NOT NULL;

In [0]:
%sql
SELECT * FROM customer_base LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW customers_norm AS
SELECT 
  customer_id,
  customer_unique_id,
  CAST(customer_zip_code_prefix AS INT) AS customer_zip_code_prefix,
  LOWER(TRIM(regexp_replace(customer_city, '\s+', ' '))) AS customer_city,
  UPPER(TRIM(regexp_replace(customer_state, '\s+', ' '))) AS customer_state
FROM customer_base;

In [0]:
%sql
CREATE OR REPLACE TABLE silver_olist.customers
USING DELTA
AS
SELECT
  customer_id,
  customer_unique_id,
  customer_zip_code_prefix,
  customer_city,
  customer_state
FROM customers_norm;

In [0]:
%sql
COMMENT ON TABLE course_training_catalog.silver_olist.customers IS
'Source: bronze_olist.customers
Silver transformations: PK hygiene (DISTINCT, non-null), city lowercase+trim, state uppercase+trim, Zip INT cast
Purpose: clean, analysis-ready customer dimension'

In [0]:
%sql
DESCRIBE EXTENDED course_training_catalog.silver_olist.customers;

In [0]:
%sql
SELECT COUNT(*) FROM course_training_catalog.silver_olist.customers;

## Cleaning and Normalizing Sellers table

In [0]:
%sql
-- create a temporary table that excludes rows with mising seller ID values
CREATE OR REPLACE TEMP VIEW sellers_nn AS
SELECT
  seller_id,
  seller_zip_code_prefix,
  seller_city,
  seller_state
FROM course_training_catalog.bronze_olist.sellers
WHERE seller_id IS NOT NULL;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEw sellers_base AS
SELECT * 
FROM SELLERS_NN
QUALIFY ROW_NUMBER() OVER(
  PARTITION BY seller_id
  ORDER BY
    seller_city ASC NULLS LAST,
    seller_state ASC NULLS LAST,
    seller_zip_code_prefix ASC NULLS LAST
)

In [0]:
# in the above two cells, we used SQL to perform transform operations on the 
# seller's tabke, now we will switch to Python and use spark API's
from pyspark.sql import functions as F

In [0]:
df_base = spark.table("sellers_base")
display(df_base)

In [0]:
# white space cleanup and letter case standardization
df_norm = (df_base.withColumn("seller_zip_code_prefix", F.col("seller_zip_code_prefix").cast("int")).withColumn("seller_city", F.trim(F.lower(F.regexp_replace(F.col("seller_city"), r"\s+", " ")))).withColumn("seller_state", F.trim(F.upper(F.regexp_replace(F.col("seller_state"), r"\s+", " ")))))
display(df_norm)

In [0]:
df_norm.select("seller_id","seller_zip_code_prefix","seller_city","seller_state").write.mode("overwrite").format("delta").saveAsTable("course_training_catalog.silver_olist.sellers")

In [0]:
%sql
-- A quick check of the above sellers table in silver layer using sql
SELECT
  COUNT(*) AS n_rows,
  COUNT(DISTINCT seller_id) AS unique_ids,
  COUNT(*) - COUNT(DISTINCT seller_id) AS dupes,
  COUNT(CASE WHEN seller_id IS NULL THEN 1 END) AS null_ids
FROM course_training_catalog.silver_olist.sellers;)


In [0]:
%sql
COMMENT ON TABLE course_training_catalog.silver_olist.sellers IS
'Source: course_training_catalog.bronze_olist.sellers
Silver transformations: PK hygiene (ROW_NUMBER per seller_id), city/state normalization, zip INT cast. 
Note: Profiel hinted dupes; exact SQL found none. DISTINCT kept as hygiene check'

In [0]:
%sql
DESCRIBE EXTENDED silver_olist.sellers;

In [0]:
base_path = "/Volumes/course_training_catalog/bronze_olist/olist_volumes"
#Orders
df_orders = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(base_path+"/orders/orders.csv")
#Customers
df_customers = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(f"{base_path}/customers/customers.csv")
#Sellers
df_sellers = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(base_path+"/sellers/sellers.csv")
#Products
df_products = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(base_path+"/products/products.csv")
# Product Category Translation
df_categories = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(base_path+"/products/product_category_name_translation.csv") 
#Order Items
df_order_items = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(base_path+"/orders/order_items.csv")
#Order Payments
df_order_payments = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(base_path+"/orders/order_payments.csv")
#Order Review Ratings
df_order_reviews = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(base_path+"/orders/order_reviews.csv")
#Geolocation
df_geolocation = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(base_path+"/customers/geolocation.csv")

#########################################################################################################

In [0]:
# we will start with products table in this notebook; customers and sellers were covered in the earlier notebook
display(df_products)

In [0]:
%sql
-- Step1. we will deal with type conversion here
CREATE OR REPLACE TEMP VIEW product_cast AS
SELECT
  product_id,
  product_category_name,
  CAST(product_name_lenght AS INT) AS product_name_lenght,
  CAST(product_description_lenght AS INT) AS product_description_lenght,
  CAST(product_photos_qty AS INT) AS product_photos_qty,
  CAST(product_weight_g AS FLOAT) AS product_weight_g,
  CAST(product_length_cm AS FLOAT) AS product_length_cm,
  CAST(product_height_cm AS FLOAT) AS product_height_cm,
  CAST(product_width_cm AS FLOAT) AS product_width_cm
FROM course_training_catalog.bronze_olist.products

In [0]:
%sql
SELECT * FROM product_cast LIMIT 10;

In [0]:
%sql
-- Step 2. Outlier removal
-- we will assume that the product weight should be greater than zero and less than 10,000 g
-- we will assume that the product dimensions shoulbe be in between 0 and 200 cms
CREATE OR REPLACE TEMP VIEW products_clean AS
SELECT *
FROM product_cast
WHERE 
  product_weight_g > 0 AND product_weight_g < 10000 AND 
  product_length_cm > 0 AND product_length_cm < 200 AND 
  product_height_cm > 0 AND product_height_cm < 200 AND
  product_width_cm > 0 AND product_width_cm < 200;

In [0]:
%sql
SELECT * FROM products_clean LIMIT 10

In [0]:
%sql
-- handle missing values in the products table
-- COALESCE function replaces null values in the specified column with the value we specify
CREATE OR REPLACE TEMP VIEW p1 AS
SELECT
  product_id,
  COALESCE(product_category_name, "unknown_category") AS product_category_name,
  COALESCE(product_photos_qty, 1) AS product_photos_qty, -- replaces null values with 1
  product_name_lenght,
  product_description_lenght,
  product_weight_g,
  product_length_cm,
  product_height_cm,
  product_width_cm
FROM products_clean;
    
SELECT * FROM p1 LIMIT 10


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW products_filled AS
SELECT
  product_id,
  product_category_name,
  product_photos_qty,
  COALESCE(product_name_lenght,CAST(ROUND(AVG(product_name_lenght) OVER (PARTITION BY product_category_name)) AS INT)) AS product_name_lenght,
  COALESCE(product_description_lenght,CAST(ROUND(AVG(product_description_lenght) OVER (PARTITION BY product_category_name)) AS INT)) AS product_description_lenght,
  product_weight_g,
  product_length_cm,
  product_height_cm,
  product_width_cm
FROM p1;


In [0]:
%sql
SELECT * FROM products_filled LIMIT 10;

In [0]:
%sql
-- create derived columns - featured engineering
CREATE OR REPLACE TEMP VIEW products_enriched AS
SELECT 
  p.*,
  CAST(ROUND(p.product_length_cm * p.product_height_cm * p.product_width_cm, 2) AS DECIMAL(18,2)) AS product_volume_cm3,
  CAST(ROUND(
    CASE WHEN(p.product_length_cm * p.product_height_cm * p.product_width_cm) > 0 THEN
      p.product_weight_g / (p.product_length_cm * p.product_height_cm * p.product_width_cm)
      ELSE NULL END, 4) AS DECIMAL(18,4)) AS product_density_g_per_cm3
FROM products_filled p

In [0]:
%sql
SELECT * FROM products_enriched LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE course_training_catalog.silver_olist.products AS
SELECT
  product_id,
  product_category_name,
  product_name_lenght,
  product_description_lenght,
  product_photos_qty,
  product_weight_g,
  product_length_cm,
  product_height_cm,
  product_width_cm,
  product_volume_cm3,
  product_density_g_per_cm3,
  current_timestamp() AS last_updated_ts
FROM products_enriched;

In [0]:
%sql
SELECT * FROM course_training_catalog.silver_olist.products LIMIT 10;

## Cleaning and Normalizing the Orders table

In [0]:
display(df_orders) # starting with the orders table from here

In [0]:
%sql
CREATE or REPLACE TEMP VIEW v1_orders_base AS
SELECT *
FROM course_training_catalog.bronze_olist.orders;

In [0]:
%sql
CREATE or REPLACE TEMP VIEW v2_orders_dates AS
SELECT
  *,
  TO_DATE(order_purchase_timestamp) AS purchase_date,
  TO_DATE(order_approved_at) AS approved_date,
  TO_DATE(order_delivered_carrier_date) AS carrier_date,
  TO_DATE(order_delivered_customer_date) AS delivered_date,
  TO_DATE(order_estimated_delivery_date) AS estimated_date
FROM v1_orders_base;

In [0]:
%sql
SELECT * FROM v2_orders_dates LIMIT 10;

In [0]:
%sql
CREATE or REPLACE TEMP VIEW v3_orders_timeparts AS
SELECT 
  *,
  HOUR(order_purchase_timestamp) AS purchase_hour,
  DAYOFWEEK(order_purchase_timestamp) AS purchase_weekday,
  DATE_FORMAT(order_purchase_timestamp, "E") AS purchase_weekday_name,
  CASE WHEN order_delivered_customer_date IS NOT NULL THEN HOUR(order_delivered_customer_date) END AS delivered_hour,
  CASE WHEN order_delivered_customer_date IS NOT NULL THEN DAYOFWEEK(order_delivered_customer_date) END AS delivered_weekday,
  CASE WHEN order_delivered_customer_date IS NOT NULL THEN DATE_FORMAT(order_delivered_customer_date, "E") END AS delivered_weekday_name
FROM v2_orders_dates;

In [0]:
%sql
SELECT * FROM v3_orders_timeparts LIMIT 10;
    

In [0]:
%sql
-- number of days between when the order was placed and when it was delivered to the customer.
CREATE or REPLACE TEMP VIEW v4_orders_metrics AS
SELECT
  *,
  DATEDIFF(order_delivered_customer_date, order_purchase_timestamp) AS delivery_days, -- number of days it took from order puchase to delivery
  CASE WHEN order_delivered_customer_date IS NOT NULL THEN DATEDIFF(order_delivered_customer_date, order_estimated_delivery_date) END AS delay_days -- difference between estimated and actual delivery
  FROM v3_orders_timeparts;
    
SELECT * FROM v4_orders_metrics LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW v5_orders_dayparts AS
SELECT
  *,
  CASE 
    WHEN purchase_hour BETWEEN 6 AND 11 THEN 'Morning'
       WHEN purchase_hour BETWEEN 12 AND 17 THEN 'Afternoon'
       WHEN purchase_hour BETWEEN 18 AND 23 THEN 'Evening'
       ELSE "night" END AS purchase_part_of_day,
  CASE WHEN purchase_weekday IN (1,7) THEN TRUE ELSE FALSE END AS is_weekend_purchase
FROM v4_orders_metrics;

In [0]:
%sql
SELECT * FROM v5_orders_dayparts LIMIT 10;


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW v6_orders_status AS
SELECT 
  *,
  LOWER(TRIM(order_status)) AS order_status_norm,
  CASE
    WHEN LOWER(TRIM(order_status)) IN ('delivered', 'invoiced') THEN 'completed'
    WHEN LOWER(TRIM(order_status)) IN ('canceled', 'unavailable') THEN 'cancelled'
    ELSE 'pending'
  END AS order_status_group
  FROM v5_orders_dayparts;
    
SELECT * FROM v6_orders_status LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW v7_orders_quality AS
SELECT
  *,
  (
    (order_approved_at < order_purchase_timestamp) OR 
    (order_delivered_carrier_date < order_approved_at) OR 
    (order_delivered_customer_date < order_delivered_carrier_date)
  ) AS is_invalid_timeflow 
  FROM v6_orders_status;
    
SELECT * FROM v7_orders_quality LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE course_training_catalog.silver_olist.orders AS
SELECT 
  order_id,
  customer_id,
  order_status_group,
  order_purchase_timestamp,
  order_delivered_customer_date,
  purchase_date,
  approved_date,
  carrier_date,
  delivered_date,
  estimated_date,
  purchase_hour,
  delivered_hour,
  purchase_weekday_name,
  delivered_weekday_name,
  delivery_days,
  delay_days,
  purchase_part_of_day,
  is_weekend_purchase,
  is_invalid_timeflow
FROM v7_orders_quality

In [0]:
%sql
SELECT * FROM course_training_catalog.silver_olist.orders LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE course_training_catalog.silver_olist.orders AS
SELECT
  *,
  CASE WHEN order_delivered_customer_date IS NULL THEN TRUE ELSE FALSE END AS is_missing_customer_date,
  CASE WHEN carrier_date is NULL THEN TRUE ELSE FALSE END AS is_missing_carrier_date,
  CASE WHEN approved_date IS NULL THEN TRUE ELSE FALSE END AS is_missing_approved_date
FROM course_training_catalog.silver_olist.orders;
    
SELECT * FROM course_training_catalog.silver_olist.orders LIMIT 10;

## Order_Items data transformation and quality checks
we will use pyspark for data transformations

In [0]:
display(df_order_items)

In [0]:
from pyspark.sql import functions as F, Window

CATALOG = "course_training_catalog"
SILVER = "silver_olist"
TGT_TABLE = f"{CATALOG}.{SILVER}.order_items"

In [0]:
df_base = df_order_items.select(
    F.trim("order_id").alias("order_id"),
    F.col("order_item_id").cast("int").alias("order_item_id"),
    F.trim("product_id").alias("product_id"),
    F.trim("seller_id").alias("seller_id"),
    F.to_timestamp("shipping_limit_date").alias("shipping_limit_date"),
    F.col("price").cast("double").alias("price"),
    F.col("freight_value").cast("double").alias("freight_value")
)

display(df_base.limit(5))

In [0]:
# Removing dupplicates
# we will define a window function that can divide the order items into small groups. 
# note that backslash(\) in python means that the next line is a continuation of the current line
# Also, the order by in the first window means that if the same item appears multiple times, pick the # one with the newest date and if the dates are equal, pick the one with the highest price. 
w = Window.partitionBy("order_id", "order_item_id") \
    .orderBy(F.col("shipping_limit_date").desc_nulls_last(), F.col("price").desc_nulls_last())

""" for the 'rn' (row number) column, we are using the window function row_number() to assign a unique number to each row in the partition, this will assign a rown number to each record. Next we apply the over function and pass in the window variable. According the window rules, spark will assign row numbers. So, we only pick up the most recent one from each row (most recent date and highest price)
"""

df_nodup = (
    df_base.withColumn("rn", F.row_number().over(w)).where(F.col("rn") == 1).drop("rn")
)


In [0]:
display(df_nodup)

In [0]:
df_flagged = (
    df_nodup
    .withColumn("is_price_nonpositive", F.col("price") <= 0)
    .withColumn("is_freight_nonpositive", F.col("freight_value") < 0)
    .withColumn(
        "is_ship_date_outlier", 
        ((F.year("shipping_limit_date") < F.lit(2017)) | (F.year("shipping_limit_date") > F.lit(2019)))
    )
)

In [0]:
display(df_flagged)

In [0]:
df_silver = (
    df_flagged
    .withColumn("total_item_value", F.round(F.col("price") + F.col("freight_value"), 2))
    .withColumn(
        "freight_ratio",
        F.when(F.col("price") > 0, F.round(F.col("freight_value") / F.col("price"), 3))
        .otherwise(F.lit(None))
    )
    .withColumn("ship_year", F.year("shipping_limit_date"))
    .withColumn("ship_month", F.month("shipping_limit_date"))
    .withColumn("ship_day", F.dayofmonth("shipping_limit_date"))
)

In [0]:
display(df_silver)

In [0]:
(df_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TGT_TABLE)
)

In [0]:
%sql
SELECT * FROM course_training_catalog.silver_olist.order_items LIMIT 10;